# 1. Setup & Environment
- 공간 통계 및 공간 데이터 연산 라이브러리(GeoPandas, PySAL, ESDA) 로드
- 부동소수점 출력 포맷 및 판다스 디스플레이 환경 설정

In [1]:
# 1. 라이브러리 로드 및 환경 설정
from pathlib import Path
import warnings

import esda
import geopandas as gpd
import libpysal
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda x: "%.4f" % x)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)

# 2. Configuration & Spatial Analysis Parameters
- 집계구 폴리곤, 생활인구, 접근성 산출물 디렉터리 경로 설정
- 공간가중치(KNN k=8), 유의수준(p<0.05), 서울시 구 매핑 및 5대 확정 시나리오 정의

In [2]:
# 2. 경로 및 분석 파라미터 설정
BASE_DIR = Path("/mnt/cowork/EV")
BOUNDARY_FP = BASE_DIR / "input/raw/집계구_2016/집계구.shp"
D1_FP = BASE_DIR / "input/processed/서울시_생활인구/집계구_생활인구_원본(OA-14979)/d1_final_2021_2024.csv"

DIR_GAUSSIAN = BASE_DIR / "output/g2sfca_sfast_final_gaussian"
DIR_SIMYA_GAUSSIAN = BASE_DIR / "output/g2sfca_sfast_simya_gaussian"
DIR_OUTPUT = BASE_DIR / "output"
DIR_OUTPUT.mkdir(parents=True, exist_ok=True)

# 모델링 파라미터
K_NEIGHBORS = 8
P_THRESHOLD = 0.05
YEARS = [2021, 2022, 2023, 2024]

# 서울시 시군구 코드 매핑 테이블
GU_MAP = {
    "11010": "종로구", "11020": "중구", "11030": "용산구", "11040": "성동구", "11050": "광진구",
    "11060": "동대문구", "11070": "중랑구", "11080": "성북구", "11090": "강북구", "11100": "도봉구",
    "11110": "노원구", "11120": "은평구", "11130": "서대문구", "11140": "마포구", "11150": "양천구",
    "11160": "강서구", "11170": "구로구", "11180": "금천구", "11190": "영등포구", "11200": "동작구",
    "11210": "관악구", "11220": "서초구", "11230": "강남구", "11240": "송파구", "11250": "강동구",
}

# 1) 확정 시나리오 5개 (지속성 집계 대상: 5개 시나리오 x 4개년 = 총 20개 조합)
CONFIRMED_SCENARIOS = [
    ("week_오전_congested", "오전_avg", "평일오전(congested)"),
    ("week_낮_normal", "낮_avg", "평일낮(normal)"),
    ("weekend_오전_freeflow", "오전_avg", "주말오전(freeflow)"),
    ("weekend_낮_normal", "낮_avg", "주말낮(normal)"),
    ("week_심야_freeflow", "심야_avg", "평일심야(freeflow)"),
]

# 2) 탐색적 시나리오 (지속성 집계 제외)
EXPLORATORY_SCENARIOS = [
    ("weekend_심야_freeflow", "심야_avg", "주말심야(탐색적)"),
]

print(f">> 총 분석 연도: {len(YEARS)}개년 | 확정 시나리오: {len(CONFIRMED_SCENARIOS)}개 | 총 지속성 조합: {len(YEARS) * len(CONFIRMED_SCENARIOS)}회")

>> 총 분석 연도: 4개년 | 확정 시나리오: 5개 | 총 지속성 조합: 20회


# 3. Spatial Weights & Hotspot Analytics Engine
- 집계구 중심점 기반 KNN(k=8) 공간가중행렬 구축 함수
- Local Getis-Ord Gi*(이진 가중) 및 고수요 콜드스팟 우선후보 판정 함수

In [3]:
# 3. 공간 분석 엔진 정의
def build_knn_weights(gdf: gpd.GeoDataFrame, k: int = K_NEIGHBORS) -> libpysal.weights.KNN:
    """집계구 Centroid 기준 KNN 공간가중행렬 생성"""
    centroids = np.array([[geom.centroid.x, geom.centroid.y] for geom in gdf.geometry])
    return libpysal.weights.KNN.from_array(centroids, k=k)


def compute_gi_star_priority(
    y_scores: np.ndarray,
    w_spatial: libpysal.weights.W,
    demand_series: pd.Series,
    oa_codes: pd.Series,
    p_th: float = P_THRESHOLD
) -> tuple[np.ndarray, np.ndarray]:
    """단일 시나리오 대상 Local Gi* 산출 및 우선설치 후보지역 판정"""
    # 1. Binary 공간가중 변환 기반 Local Gi* 산출
    lg = esda.getisord.G_Local(y_scores, w_spatial, transform="B")
    
    # 2. 통계적 유의성(p < 0.05) 기반 Hot/Cold Spot 분류
    coded = np.where(
        (lg.Zs < 0) & (lg.p_norm < p_th), "Cold Spot",
        np.where((lg.Zs > 0) & (lg.p_norm < p_th), "Hot Spot", "Not Sig")
    )
    
    # 3. 고수요(생활인구 >= 중위값) 콜드스팟 우선설치 후보 판정
    demand_vals = oa_codes.map(demand_series).fillna(0.0).values
    valid_demands = demand_vals[demand_vals > 0]
    median_demand = np.median(valid_demands) if len(valid_demands) > 0 else 0.0
    
    is_priority = (coded == "Cold Spot") & (demand_vals >= median_demand)
    return coded, is_priority

# 4. Data Loaders & Score Path Resolver
- 서울시 집계구 폴리곤(EPSG:5179) 로더 및 공간 전처리
- 신규 산출물(`_mw.csv`) 및 원본 산출물 파일 경로 자동 매퍼

In [4]:
# 4. 데이터 로더 및 파일 경로 매퍼
def load_seoul_boundary(boundary_path: Path) -> gpd.GeoDataFrame:
    """서울시 집계구(코드 11 시작) 경계 로드 및 좌표계 설정"""
    gdf = gpd.read_file(boundary_path)
    gdf = gdf.set_crs(epsg=5179, allow_override=True)
    gdf["TOT_REG_CD"] = gdf["TOT_REG_CD"].astype(str)
    gdf = gdf[gdf["TOT_REG_CD"].str.startswith("11")].copy().reset_index(drop=True)
    return gdf


def resolve_score_file(year: int, suffix_tag: str) -> Path:
    """접근성 점수 파일 경로 탐색 (_mw 우선, 구버전 호환)"""
    fp_new = DIR_GAUSSIAN / f"g2sfca_score_{year}_{suffix_tag}_mw.csv"
    if fp_new.exists():
        return fp_new
    
    fp_orig = DIR_GAUSSIAN / f"g2sfca_score_{year}_{suffix_tag}.csv"
    if fp_orig.exists():
        return fp_orig
    
    if "심야" in suffix_tag:
        clean_tag = "week_심야" if "week" in suffix_tag else "weekend_심야"
        fp_simya_mw = DIR_GAUSSIAN / f"g2sfca_score_{year}_{clean_tag}_mw.csv"
        if fp_simya_mw.exists():
            return fp_simya_mw
        fp_simya_orig = DIR_SIMYA_GAUSSIAN / f"g2sfca_score_{year}_{clean_tag}.csv"
        if fp_simya_orig.exists():
            return fp_simya_orig

    return None

# 5. Spatial Weights Construction & Demand Prep
- 서울시 14,979개 집계구 폴리곤 로드 및 KNN(k=8) 공간가중행렬 생성
- 시계열 생활인구 데이터 로드

In [5]:
# 5. 집계구 경계 로드 및 공간가중행렬 구축
gdf_seoul = load_seoul_boundary(BOUNDARY_FP)
print(f">> 서울시 집계구 경계 로드 완료: 총 {len(gdf_seoul):,}개 집계구")

w_knn = build_knn_weights(gdf_seoul, k=K_NEIGHBORS)
print(f">> KNN 공간가중행렬(k={K_NEIGHBORS}) 구축 완료 (Non-zero weights: {w_knn.nonzero:,}개)")

df_pop_all = pd.read_csv(D1_FP, dtype={"집계구코드": str})
print(f">> 생활인구 데이터 로드 완료: 총 {len(df_pop_all):,}개 행")

>> 서울시 집계구 경계 로드 완료: 총 19,153개 집계구
>> KNN 공간가중행렬(k=8) 구축 완료 (Non-zero weights: 153,224개)
>> 생활인구 데이터 로드 완료: 총 76,612개 행


# 6. Batch Hotspot & Persistence Computation Engine
- 20대 확정 시나리오 대상 Local Gi* 연산 및 시계열 지속성(0~20점) 누적
- 주말 심야 탐색적 시나리오 별도 진단

In [6]:
# 6. 핫스팟 연산 및 지속성(Persistence) 누적 실행
print("=" * 85)
print("RUNNING: GETIS-ORD GI* HOTSPOT & PERSISTENCE DIAGNOSIS PIPELINE")
print("=" * 85)

persistence_series = pd.Series(0, index=gdf_seoul["TOT_REG_CD"])
n_confirmed_combos = 0

# 1. 확정 시나리오 일괄 분석 (지속성 스코어 집계 대상)
print("\n--- [1] 확정 시나리오 분석 (지속성 스코어 산출 대상) ---")
for suffix_tag, pop_col, label in CONFIRMED_SCENARIOS:
    for year in YEARS:
        fp_score = resolve_score_file(year, suffix_tag)
        if fp_score is None:
            print(f"  [!] 점수 파일 누락 스킵: {year} | {suffix_tag}")
            continue
            
        df_acc = pd.read_csv(fp_score, dtype={"oa_code": str})
        df_acc_reindexed = df_acc.set_index("oa_code").reindex(gdf_seoul["TOT_REG_CD"]).reset_index()
        y_scores = df_acc_reindexed["accessibility_score"].fillna(0.0).values
        
        df_pop_y = df_pop_all[df_pop_all["year"] == year].set_index("집계구코드")[pop_col]
        
        coded, is_priority = compute_gi_star_priority(
            y_scores, w_knn, df_pop_y, gdf_seoul["TOT_REG_CD"], p_th=P_THRESHOLD
        )
        
        n_confirmed_combos += 1
        persistence_series.loc[is_priority] += 1
        
        n_cold = (coded == "Cold Spot").sum()
        n_hot = (coded == "Hot Spot").sum()
        n_prio = is_priority.sum()
        print(f"  [>] {label:<22} {year} | Cold: {n_cold:4d} | Hot: {n_hot:4d} | 우선후보: {n_prio:4d}")

print(f"\n>> 총 누적 확정 조합 수: {n_confirmed_combos}개 (5개 시나리오 x {len(YEARS)}개년)")

# 2. 탐색적 시나리오 별도 분석 (지속성 집계 제외)
print("\n--- [2] 탐색적 시나리오 분석 (지속성 집계 제외) ---")
for suffix_tag, pop_col, label in EXPLORATORY_SCENARIOS:
    for year in YEARS:
        fp_score = resolve_score_file(year, suffix_tag)
        if fp_score is None:
            continue
            
        df_acc = pd.read_csv(fp_score, dtype={"oa_code": str})
        df_acc_reindexed = df_acc.set_index("oa_code").reindex(gdf_seoul["TOT_REG_CD"]).reset_index()
        y_scores = df_acc_reindexed["accessibility_score"].fillna(0.0).values
        df_pop_y = df_pop_all[df_pop_all["year"] == year].set_index("집계구코드")[pop_col]
        
        coded, is_priority = compute_gi_star_priority(
            y_scores, w_knn, df_pop_y, gdf_seoul["TOT_REG_CD"], p_th=P_THRESHOLD
        )
        
        n_cold = (coded == "Cold Spot").sum()
        n_hot = (coded == "Hot Spot").sum()
        n_prio = is_priority.sum()
        print(f"  [>] {label:<22} {year} | Cold: {n_cold:4d} | Hot: {n_hot:4d} | 우선후보: {n_prio:4d}")

RUNNING: GETIS-ORD GI* HOTSPOT & PERSISTENCE DIAGNOSIS PIPELINE

--- [1] 확정 시나리오 분석 (지속성 스코어 산출 대상) ---
  [>] 평일오전(congested)        2021 | Cold: 5747 | Hot: 4927 | 우선후보: 2819
  [>] 평일오전(congested)        2022 | Cold: 5838 | Hot: 4660 | 우선후보: 2846
  [>] 평일오전(congested)        2023 | Cold: 5405 | Hot: 4069 | 우선후보: 2728
  [>] 평일오전(congested)        2024 | Cold: 5598 | Hot: 3972 | 우선후보: 2774
  [>] 평일낮(normal)            2021 | Cold: 5586 | Hot: 4476 | 우선후보: 2852
  [>] 평일낮(normal)            2022 | Cold: 5110 | Hot: 3908 | 우선후보: 2637
  [>] 평일낮(normal)            2023 | Cold: 5752 | Hot: 4504 | 우선후보: 3027
  [>] 평일낮(normal)            2024 | Cold: 5864 | Hot: 4415 | 우선후보: 3055
  [>] 주말오전(freeflow)         2021 | Cold: 4716 | Hot: 5241 | 우선후보: 2290
  [>] 주말오전(freeflow)         2022 | Cold: 4751 | Hot: 4996 | 우선후보: 2278
  [>] 주말오전(freeflow)         2023 | Cold: 5670 | Hot: 5913 | 우선후보: 2965
  [>] 주말오전(freeflow)         2024 | Cold: 6019 | Hot: 5407 | 우선후보: 3149
  [>] 주말낮(normal)            202

# 7. Summary Analytics, Top Persistent Dong Ranking & Export
- 지속성 만점(20/20) 최우선 설치 후보지역 도출
- 행정동별 최다 후보지 랭킹 출력 및 지속성 진단 결과 파일(`_mw.csv`) 저장

In [7]:
# 7. 결과 집계, 동별 랭킹 산출 및 CSV 저장
df_result = pd.DataFrame({
    "oa_code": gdf_seoul["TOT_REG_CD"],
    "gu": gdf_seoul["TOT_REG_CD"].str[:5].map(GU_MAP),
    "dong": gdf_seoul["ADM_NM"],
    "persistence": persistence_series.values,
    "n_combos": n_confirmed_combos,
})

# 결과 파일 저장 (_mw)
out_fp = DIR_OUTPUT / "hotspot_persistence_final_mw.csv"
df_result.to_csv(out_fp, index=False, encoding="utf-8-sig")
print(f">> 지속성 진단 결과 저장 완료: {out_fp} (총 {len(df_result):,}개 집계구)")

# 지속성 만점(20/20) 집계구 동별 랭킹 요약
full_persistent = df_result[df_result["persistence"] == n_confirmed_combos]
print("\n" + "=" * 80)
print(f"       20대 조합 전수({n_confirmed_combos}/{n_confirmed_combos}) 일관 우선설치 후보 집계구: 총 {len(full_persistent)}개")
print("=" * 80)

dong_ranking = (
    full_persistent.groupby(["gu", "dong"])
    .size()
    .reset_index(name="persistent_oa_count")
    .sort_values(by="persistent_oa_count", ascending=False)
)

print("\n=== [최우선 설치 필요 행정동 상위 15개 순위] ===")
display(dong_ranking.head(15).reset_index(drop=True))

>> 지속성 진단 결과 저장 완료: /mnt/cowork/EV/output/hotspot_persistence_final_mw.csv (총 19,153개 집계구)

       20대 조합 전수(20/20) 일관 우선설치 후보 집계구: 총 113개

=== [최우선 설치 필요 행정동 상위 15개 순위] ===


,gu,dong,persistent_oa_count
0,강동구,길동,20
1,강동구,천호1동,16
2,강동구,천호2동,12
3,관악구,난곡동,10
4,강동구,암사1동,7
5,관악구,난향동,6
6,강동구,둔촌2동,5
7,강동구,명일1동,5
8,동작구,사당3동,5
9,구로구,개봉3동,4


# 8. Result Validation (Comparison with Original Outputs)
- 원본 지속성 진단 결과(`hotspot_persistence_final.csv`)와 신규 산출물(`_mw.csv`) 간 수치 일치성 검증

In [8]:
# 8. 원본 산출물 vs 신규 산출물(_mw) 정밀 오차 검증
fp_orig = DIR_OUTPUT / "hotspot_persistence_final.csv"
fp_new = DIR_OUTPUT / "hotspot_persistence_final_mw.csv"

if not fp_orig.exists():
    print(f"[!] 비교할 원본 결과 파일이 존재하지 않습니다: {fp_orig}")
elif not fp_new.exists():
    print(f"[!] 신규 산출물 파일이 생성되지 않았습니다: {fp_new}")
else:
    df_orig = pd.read_csv(fp_orig, dtype={"oa_code": str})
    df_new = pd.read_csv(fp_new, dtype={"oa_code": str})
    
    comp = df_orig.merge(df_new, on="oa_code", suffixes=("_orig", "_new"))
    comp["diff_persistence"] = (comp["persistence_orig"] - comp["persistence_new"]).abs()
    
    max_diff = comp["diff_persistence"].max()
    match_rate = (comp["diff_persistence"] == 0).mean() * 100
    
    print("=" * 80)
    print("           핫스팟 지속성 진단 원본 vs 리팩토링 코드 수치 대조 요약표")
    print("=" * 80)
    print(f"- 총 비교 집계구 수 : {len(comp):,}개")
    print(f"- 지속성 점수 일치율: {match_rate:.2f}%")
    print(f"- 최대 차이 (Max Diff): {max_diff}")
    
    if max_diff == 0:
        print(">> [판정] 모든 집계구의 지속성 점수(Persistence)가 원본 산출물과 100% 완벽히 일치합니다.")
    else:
        print(">> [판정] 심야 윈도우 기준 변경(5시간) 등으로 인한 일부 집계구의 미세 차이가 반영되었습니다.")
        print("\n[차이가 발생한 집계구 샘플]")
        display(comp[comp["diff_persistence"] > 0].head(5))

           핫스팟 지속성 진단 원본 vs 리팩토링 코드 수치 대조 요약표
- 총 비교 집계구 수 : 19,153개
- 지속성 점수 일치율: 99.44%
- 최대 차이 (Max Diff): 2
>> [판정] 심야 윈도우 기준 변경(5시간) 등으로 인한 일부 집계구의 미세 차이가 반영되었습니다.

[차이가 발생한 집계구 샘플]


,oa_code,gu_orig,dong_orig,persistence_orig,n_combos_orig,gu_new,dong_new,persistence_new,n_combos_new,diff_persistence
19,1101054010004,종로구,삼청동,7,20,종로구,삼청동,8,20,1
52,1101056020018,종로구,평창동,16,20,종로구,평창동,17,20,1
190,1101069010101,종로구,창신3동,1,20,종로구,창신3동,2,20,1
244,1101072010002,종로구,청운효자동,4,20,종로구,청운효자동,5,20,1
257,1101073010012,종로구,혜화동,17,20,종로구,혜화동,18,20,1
